# Logistic Regression
**Project:** Hotel Booking Demand - Supervised Learning Assignment  
**Task:** Predict hotel booking cancellations using **Logistic Regression**.

## 1. Imports

In [ ]:
# ============================================================
    # Cell 1: Import model-specific libraries and helper functions
    # ============================================================
    from pathlib import Path
    import pandas as pd
    from IPython.display import display, Markdown
    from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, StratifiedKFold

    from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

    from hotel_booking_common import (
        RANDOM_STATE,
        get_project_root,
        ensure_dir,
        load_dataset,
        create_dataset_description_bundle,
        prepare_hotel_booking_dataframe,
        summarize_preprocessing_notes,
        build_train_test_split,
        get_feature_groups,
        build_preprocessor,
        evaluate_classifier,
        save_cv_results,
        save_feature_insights,
        save_model_bundle,
        save_json,
    )

## 2. Project configuration

In [ ]:
# ============================================================
# Cell 2: Configure dataset path and model-specific output folder
# ============================================================
PROJECT_ROOT = get_project_root()
DATASET_DIR = PROJECT_ROOT / "dataset"
OUTPUT_DIR = ensure_dir(PROJECT_ROOT / "outputs" / "logistic_regression")

print(f"Project root: {PROJECT_ROOT}")
print(f"Dataset directory: {DATASET_DIR}")
print(f"Output directory: {OUTPUT_DIR}")

## 3. Load dataset

In [ ]:
# ============================================================
# Cell 3: Load the raw dataset from the root-level dataset folder
# ============================================================
df_raw, dataset_path = load_dataset(DATASET_DIR)

print(f"Dataset file: {dataset_path.name}")
print(f"Raw shape: {df_raw.shape}")
display(df_raw.head())

## 4. Dataset description and exploratory analysis

In [ ]:
# ============================================================
# Cell 4: Save a dataset description bundle inside this model folder
# ============================================================
eda_bundle = create_dataset_description_bundle(
    df=df_raw,
    output_dir=OUTPUT_DIR,
    prefix="logistic_regression_dataset",
    full=True,
)

display(Markdown("### Overview"))
display(pd.DataFrame([eda_bundle["overview"]]))

## 5. Data preprocessing and feature engineering

In [ ]:
# ============================================================
# Cell 5: Apply domain-aware preprocessing before model pipelines
# ============================================================
df_model, preprocessing_notes = prepare_hotel_booking_dataframe(df_raw, target_col="is_canceled")

display(Markdown("### Preprocessing notes"))
preprocessing_notes_df = summarize_preprocessing_notes(preprocessing_notes)
display(preprocessing_notes_df)
preprocessing_notes_df.to_csv(OUTPUT_DIR / "preprocessing_summary_table.csv", index=False)

## 6. Train-test split and feature grouping

In [ ]:
# ============================================================
# Cell 6: Create a reproducible split and identify feature groups
# ============================================================
X_train, X_test, y_train, y_test = build_train_test_split(
    df_model,
    target_col="is_canceled",
    test_size=0.20,
    random_state=RANDOM_STATE,
)

feature_cols, numeric_cols, categorical_cols = get_feature_groups(df_model, target_col="is_canceled")

print(f"Training rows: {X_train.shape[0]}")
print(f"Testing rows: {X_test.shape[0]}")
print(f"Numeric features: {len(numeric_cols)}")
print(f"Categorical features: {len(categorical_cols)}")

## 7. Build model pipeline

In [ ]:
# ============================================================
    # Cell 7: Build the preprocessing pipeline and model object
    # ============================================================
    preprocessor = build_preprocessor("logistic_regression", numeric_cols, categorical_cols)


classifier = LogisticRegression(
    max_iter=2500,
    random_state=RANDOM_STATE,
)


    pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", classifier),
    ])

    display(pipeline)

## 8. Hyperparameter tuning setup

In [ ]:
# ============================================================
    # Cell 8: Configure cross-validation and hyperparameter search
    # ============================================================
    cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)


param_grid = {
    "classifier__C": [0.1, 0.5, 1.0, 2.0, 5.0],
    "classifier__solver": ["lbfgs", "liblinear"],
    "classifier__class_weight": [None, "balanced"],
}

search_object = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring="roc_auc",
    cv=cv_strategy,
    n_jobs=-1,
    verbose=1,
    refit=True,
)


    print(search_object)

## 9. Train the model

In [ ]:
# ============================================================
# Cell 9: Fit the hyperparameter search on the training data
# ============================================================
search_object.fit(X_train, y_train)

print("Best CV score:", search_object.best_score_)
print("Best parameters:", search_object.best_params_)

## 10. Save cross-validation results

In [ ]:
# ============================================================
# Cell 10: Save hyperparameter tuning results and best parameters
# ============================================================
cv_results_df = save_cv_results(search_object, OUTPUT_DIR)
display(cv_results_df.head(10))
save_json(search_object.best_params_, OUTPUT_DIR / "best_params_pretty.json")

## 11. Evaluate on the hold-out test set

In [ ]:
# ============================================================
# Cell 11: Evaluate the best model and save all test metrics
# ============================================================
best_model = search_object.best_estimator_
metrics = evaluate_classifier(
    model=best_model,
    X_test=X_test,
    y_test=y_test,
    output_dir=OUTPUT_DIR,
    model_name="Logistic Regression",
    threshold=0.50,
)

display(pd.DataFrame([metrics]))

## 12. Model interpretation

In [ ]:
# ============================================================
# Cell 12: Save coefficients or feature importances for interpretation
# ============================================================
feature_insights_df = save_feature_insights(
    model=best_model,
    output_dir=OUTPUT_DIR,
    model_name="Logistic Regression",
    top_n=20,
)

if not feature_insights_df.empty:
    display(feature_insights_df.head(20))
else:
    print("No feature importance or coefficient table is available for this model.")

## 13. Save the trained model

In [ ]:
# ============================================================
# Cell 13: Save the best trained pipeline and metadata for reuse
# ============================================================
save_model_bundle(
    model=best_model,
    output_dir=OUTPUT_DIR,
    model_filename="best_model.joblib",
    best_params=search_object.best_params_,
    preprocessing_notes=preprocessing_notes,
)

print(f"Saved model and metadata to: {OUTPUT_DIR}")

## 14. Save a short run report

In [ ]:
# ============================================================
# Cell 14: Write a short text report for quick review
# ============================================================
run_report_lines = [
    "# Logistic Regression Run Report",
    "",
    f"- Dataset file: {dataset_path.name}",
    f"- Raw dataset shape: {df_raw.shape}",
    f"- Modeling dataset shape: {df_model.shape}",
    f"- Best CV score: {search_object.best_score_:.6f}",
    f"- Best parameters: {search_object.best_params_}",
    f"- Test accuracy: {metrics['accuracy']:.6f}",
    f"- Test precision: {metrics['precision']:.6f}",
    f"- Test recall: {metrics['recall']:.6f}",
    f"- Test F1 score: {metrics['f1']:.6f}",
    f"- Test ROC-AUC: {metrics['roc_auc']:.6f}",
    f"- Test MCC: {metrics['mcc']:.6f}",
    "",
    "## Critical analysis prompts",
    "- Review whether the model overfits by comparing CV performance with test performance.",
    "- Explain how the preprocessing steps influenced model behavior.",
    "- Discuss what future feature engineering or threshold tuning could improve recall or MCC."
]

run_report_path = OUTPUT_DIR / "run_report.md"
run_report_path.write_text("\n".join(run_report_lines), encoding="utf-8")
print(f"Saved run report to: {run_report_path}")